In [11]:
import pandas as pd

ratings = pd.read_csv("../server/data/ml-latest-small/ratings.csv")
movies = pd.read_csv("../server/data/ml-latest-small/movies.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [12]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)
user_movie_matrix = user_movie_matrix.fillna(0)

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)
similar_users = user_similarity_df[1].sort_values(
    ascending=False
)

print(similar_users.head())

userId
1      1.000000
266    0.357408
313    0.351562
368    0.345127
57     0.345034
Name: 1, dtype: float64


In [ ]:
def recommend_movies(user_id, num_recommendations=5):

    similar_users = user_similarity_df[user_id]\
        .sort_values(ascending=False)[1:11]

    recommended_movies = {}

    watched_movies = ratings[
        ratings['userId'] == user_id
    ]['movieId'].values

    for similar_user, similarity_score in similar_users.items():

        user_ratings = ratings[
            ratings['userId'] == similar_user
        ]

        for _, row in user_ratings.iterrows():

            movie_id = row['movieId']
            rating = row['rating']

            # Skip already watched movies
            if movie_id in watched_movies:
                continue

            if movie_id not in recommended_movies:
                recommended_movies[movie_id] = 0

            recommended_movies[movie_id] += (
                rating * similarity_score 
            )  #calculates recommendation score

    sorted_movies = sorted(
        recommended_movies.items(),
        key=lambda x: x[1], #sort by recommendation score
        reverse=True
    )

    recommendations = []

    for movie_id, _ in sorted_movies[:num_recommendations]:

        movie_title = movies[
            movies['movieId'] == movie_id
        ]['title'].values[0]

        recommendations.append(movie_title)

    return recommendations


recommend_movies(1)

['Terminator 2: Judgment Day (1991)',
 'Aliens (1986)',
 'Sixth Sense, The (1999)',
 'Hunt for Red October, The (1990)',
 'Godfather, The (1972)']